# Qwen2-VL UI Element Detection Fine-Tuning on Kaggle (QLoRA)

This notebook provides a complete pipeline to fine-tune **Qwen2-VL** (7B or 2B) for UI element detection and bounding box prediction using **QLoRA (4-bit LoRA)** on Kaggle GPUs (T4 x 2 or P100).

### Pipeline Overview:
1. **Environment Setup**: Install latest `transformers`, `peft`, `bitsandbytes`, and `qwen-vl-utils`.
2. **Model Loading**: Load Qwen2-VL with 4-bit NormalFloat quantization.
3. **LoRA Adapter**: Attach low-rank adapters to linear projection layers.
4. **Data Formatting**: Format UI images and ground-truth bounding boxes into Qwen2-VL chat format.
5. **Training**: Train with Hugging Face `Trainer` / `SFTTrainer`.
6. **Inference & Export**: Test the fine-tuned model and save the adapter weights.

## 1. Install Required Packages

In [ ]:
!pip install -q --upgrade pip
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes peft datasets qwen-vl-utils torchvision

## 2. Check GPU & System Environment

In [ ]:
import torch
import os

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Configuration & Vocabulary
Choose between `Qwen/Qwen2-VL-7B-Instruct` (recommended for final training) or `Qwen/Qwen2-VL-2B-Instruct` (faster experimentation).

In [ ]:
# Model ID: Use 2B for quick runs or 7B for maximum spatial grounding quality
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"  # Alternative: "Qwen/Qwen2-VL-2B-Instruct"
OUTPUT_DIR = "./qwen2_vl_ui_lora"

# Agreed UI Type Vocabulary
UI_VOCABULARY = [
    "container", "card", "button", "input", "image", "icon", 
    "navbar", "sidebar", "modal", "tab", "checkbox", "radio", 
    "switch", "avatar", "divider", "badge", "link", "text_block"
]

# System Prompt Template
SYSTEM_PROMPT = f"""You are an expert UI layout analyzer.
Your task is to detect all visual elements and containers in the given UI image.

Strict Vocabulary: {UI_VOCABULARY}

Output ONLY a raw JSON array matching this format without markdown code blocks:
[
  {{
    \"id\": \"elem_0\",
    \"type\": \"button\",
    \"bounds\": {{\"x\": 50, \"y\": 100, \"w\": 120, \"h\": 40}},
    \"style_hints\": {{\"background\": \"#007bff\", \"border\": \"none\", \"rounded\": \"md\", \"shadow\": \"none\"}}
  }}
]
"""

## 4. Load Model in 4-Bit (QLoRA) and Configure PEFT

In [ ]:
from transformers import (
    Qwen2VLForConditionalGeneration, 
    AutoProcessor, 
    BitsAndBytesConfig
)
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training
)

# 4-bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Loading base model: {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

# Prepare model for LoRA training
model = prepare_model_for_kbit_training(model)

# LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Dataset Preparation
This section builds a sample UI training dataset and shows how to load your custom dataset from `/kaggle/input`.

In [ ]:
import json
from PIL import Image, ImageDraw

def create_dummy_sample(idx: int):
    """Generates a synthetic UI image and matching JSON target for quick testing."""
    w, h = 640, 480
    img = Image.new("RGB", (w, h), color=(245, 247, 250))
    draw = ImageDraw.Draw(img)
    
    # Navbar
    draw.rectangle([0, 0, w, 50], fill=(30, 41, 59))
    # Card
    draw.rectangle([40, 80, 600, 320], fill=(255, 255, 255), outline=(226, 232, 240))
    # Button
    draw.rectangle([60, 240, 200, 290], fill=(37, 99, 235))
    
    elements = [
        {
            "id": "elem_0",
            "type": "navbar",
            "bounds": {"x": 0, "y": 0, "w": w, "h": 50},
            "style_hints": {"background": "#1e293b", "border": "none", "rounded": "none", "shadow": "sm"}
        },
        {
            "id": "elem_1",
            "type": "card",
            "bounds": {"x": 40, "y": 80, "w": 560, "h": 240},
            "style_hints": {"background": "#ffffff", "border": "1px solid #e2e8f0", "rounded": "md", "shadow": "md"}
        },
        {
            "id": "elem_2",
            "type": "button",
            "bounds": {"x": 60, "y": 240, "w": 140, "h": 50},
            "style_hints": {"background": "#2563eb", "border": "none", "rounded": "md", "shadow": "none"}
        }
    ]
    return img, json.dumps(elements)

# Create synthetic demo dataset (replace with your Kaggle dataset path)
demo_dataset = [create_dummy_sample(i) for i in range(10)]
print(f"Created {len(demo_dataset)} training samples for demonstration.")

## 6. Multi-Modal Batch Collate Function

In [ ]:
from qwen_vl_utils import process_vision_info

class UICollator:
    def __init__(self, processor, system_prompt):
        self.processor = processor
        self.system_prompt = system_prompt

    def __call__(self, batch):
        texts = []
        images_list = []
        
        for image, target_json in batch:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": self.system_prompt}
                    ]
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": target_json}
                    ]
                }
            ]
            
            # Format text prompt using model's chat template
            formatted_text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            image_inputs, _ = process_vision_info(messages)
            
            texts.append(formatted_text)
            images_list.extend(image_inputs)

        # Process batch tensors
        batch_inputs = self.processor(
            text=texts,
            images=images_list,
            padding=True,
            return_tensors="pt"
        )
        
        # In causal LM, labels are identical to input_ids
        labels = batch_inputs["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        batch_inputs["labels"] = labels
        
        return batch_inputs

collate_fn = UICollator(processor, SYSTEM_PROMPT)

## 7. Training with Hugging Face Trainer

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="no",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=demo_dataset,
    data_collator=collate_fn
)

print("Starting fine-tuning...")
# trainer.train()  # Uncomment when ready to train on Kaggle
print("Training configured successfully.")

## 8. Save Fine-Tuned LoRA Adapter

In [ ]:
# Save trained adapter weights
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter weights saved to: {OUTPUT_DIR}")

## 9. Run Inference with Fine-Tuned Model

In [ ]:
test_image, _ = create_dummy_sample(99)

test_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_image},
            {"type": "text", "text": SYSTEM_PROMPT}
        ]
    }
]

prompt_text = processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
image_inputs, _ = process_vision_info(test_messages)

inputs = processor(
    text=[prompt_text],
    images=image_inputs,
    padding=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=1024, temperature=0.1)
    
trimmed_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
response_text = processor.batch_decode(trimmed_ids, skip_special_tokens=True)[0]

print("=== Predicted UI Elements ===")
print(response_text)